# NB3 — FastAPI `/search` Endpoint + Latency Benchmark

**Stack:** FastAPI + uvicorn + httpx (client). Searcher từ `app/search.py`.
Maps to slide §7 (Production Patterns) + deliverable bullets 1, 4.

> Mục tiêu: bọc `Searcher` thành REST API, đo P50/P95/P99 latency, đảm bảo
> P99 < 50 ms cho hybrid mode (rubric threshold).

In [1]:
import _setup  # noqa: F401
import json
import os as _os
import subprocess
import sys as _sys
import time
from pathlib import Path

import httpx

## 1. Khởi động API server (background)

Notebook khởi động uvicorn ở background subprocess và đợi `/healthz` trả ready.

In [2]:
ROOT = Path(_setup.__file__).resolve().parent.parent
_uvicorn = str(Path(_sys.executable).parent / "uvicorn")
_env = {**_os.environ, "QDRANT_MODE": "memory", "PYTHONIOENCODING": "utf-8"}
proc = subprocess.Popen(
    [_uvicorn, "app.main:app", "--port", "8000", "--log-level", "warning"],
    cwd=str(ROOT),
    env=_env,
)

URL = "http://localhost:8000"
# Đợi tối đa 120s — server cần ~55s để embed 1000 docs vào in-memory Qdrant
for _ in range(120):
    try:
        r = httpx.get(f"{URL}/healthz", timeout=2.0)
        if r.status_code == 200 and r.json().get("ready"):
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    proc.terminate()
    raise RuntimeError("API didn't become ready within 120s")

print(httpx.get(f"{URL}/healthz").json())

{'ready': True, 'n_docs': 1000}


## 2. Single query — kiểm tra response shape API

In [3]:
r = httpx.get(f"{URL}/search", params={"q": "cloud computing tự động mở rộng", "mode": "hybrid"})
r.raise_for_status()
body = r.json()
print(f"latency_ms: {body['latency_ms']:.1f}")
print("top-3 hits:")
for h in body["hits"][:3]:
    print(f"  {h['doc_id']:>14}  score={h['score']:.4f}  {h['title']}")

latency_ms: 13.6
top-3 hits:
       cloud_016  score=0.0325  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_072  score=0.0323  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_053  score=0.0315  Điện toán đám mây: tự động mở rộng theo lưu lượng


## 3. Stop API server

Server đã xong mục đích demo. Dừng lại trước khi chạy benchmark để tránh
port conflict và giảm memory pressure.

In [4]:
proc.terminate()
proc.wait(timeout=5)
print("API server stopped — starting standalone benchmark subprocess")

API server stopped — starting standalone benchmark subprocess


## 4. Latency benchmark (subprocess — bypass Jupyter kernel ZMQ issue)

**Tại sao subprocess?** Trên Windows, Python 3.13 Jupyter kernel dùng
`ProactorEventLoop` — khi ONNX runtime (fastembed) khởi tạo thread pool bên
trong asyncio event loop, ZMQ socket drop sau vài giây. Benchmark qua
subprocess hoàn toàn tránh vấn đề này vì ONNX chạy trong process riêng.

Script `scripts/run_nb3_benchmark.py` đo **server-side latency** identically
với cách FastAPI đo `latency_ms` — cùng `time.perf_counter()` block.

In [5]:
_bench_env = {**_os.environ, "PYTHONIOENCODING": "utf-8"}
bench_result = subprocess.run(
    [_sys.executable, str(ROOT / "scripts" / "run_nb3_benchmark.py")],
    capture_output=True,
    text=True,
    encoding="utf-8",
    timeout=300,
    env=_bench_env,
)
print(bench_result.stdout)
if bench_result.returncode != 0:
    print("STDERR:", bench_result.stderr[-500:])

Building Searcher (embed 1000 docs in-process)...
Searcher ready in 51.4s (1000 docs)

Warming up ONNX (10 hybrid queries)...
Warmup done.

=== SINGLE QUERY RESPONSE ===
latency_ms: 15.8
top-3 hits:
       cloud_016  score=0.0325  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_072  score=0.0323  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_053  score=0.0315  Điện toán đám mây: tự động mở rộng theo lưu lượng

=== LATENCY BENCHMARK (server-side, 2 reps × 50 queries) ===
  mode            P50      P95      P99
  keyword       1.9ms    4.3ms    5.7ms
  semantic      8.9ms   13.0ms   16.2ms
  hybrid       12.2ms   15.7ms   22.2ms

PASS — hybrid P99 < 50ms (22.2ms)

Results saved to D:\AI_THUCCHIEN\NGAY19\Day19-Track2-VectorFeatureStore-Lab\data\nb3_benchmark_results.json



## 5. Rubric assertion — hybrid P99 < 50ms

In [6]:
_results_path = ROOT / "data" / "nb3_benchmark_results.json"
_data = json.loads(_results_path.read_text(encoding="utf-8"))
hyb_p99 = _data["hyb_p99"]
print(f"Hybrid P99 server-side: {hyb_p99:.1f}ms")
if hyb_p99 < 50:
    print(f"PASS — hybrid P99 < 50ms ({hyb_p99:.1f}ms)")
else:
    print(f"WARN — hybrid P99 >= 50ms ({hyb_p99:.1f}ms)")

Hybrid P99 server-side: 22.2ms
PASS — hybrid P99 < 50ms (22.2ms)


## Deliverable evidence

1. Output cell 2: 1 single hybrid query response với `latency_ms` + `top-3 hits`.
2. Output cell 4: bảng P50/P95/P99 cho keyword/semantic/hybrid.
3. Output cell 5: hybrid P99 < 50ms PASS.

---

## Vibe-coding callout

**Delegate freely:** the FastAPI scaffolding (route definition, Pydantic
response model, lifespan handler). AI generates this perfectly given the
spec "GET /search?q=str&mode=Literal[...] returning SearchResponse with
latency_ms field". `app/main.py` is exactly that pattern.

**Think hard yourself:** *what to measure*. Server-side latency vs wall-clock
vs client-side. P50 vs P95 vs P99. Cold vs warm. Single user vs concurrent.
Đây là *judgement* decisions — đừng nhờ AI quyết định metric.